# Install checker log comparison

Compares the SUMMARY section of two `Test-AppInstalled` check logs.

In [1]:
import re
from pathlib import Path

LOG_DIR = Path.cwd().parent / "logs" / "check"
FIRST_LOG = LOG_DIR / "2026-09-21_21-18-55.log"
SECOND_LOG = LOG_DIR / "2026-10-05_11-19-24.log"

ITEM_RE = re.compile(r"^\s*-\s+(?P<name>.*?)\s+\[(?P<id>[^\]]+)\]\s*$")
COUNT_RE = re.compile(r"^(?P<key>[^:]+?)\s*:\s*(?P<value>\d+)\s*$")


def read_log(path):
    raw = path.read_bytes()
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw.decode("utf-16")
    return raw.decode("utf-8-sig", errors="replace")


def parse_summary(path):
    lines = read_log(path).splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip() == "SUMMARY")

    counts = {}
    lists = {"installed": {}, "stale": {}}
    current = None

    for line in lines[start + 1:]:
        stripped = line.strip()
        if stripped.startswith("Installed apps"):
            current = "installed"
            continue
        if stripped.startswith("Stale apps"):
            current = "stale"
            continue

        item = ITEM_RE.match(line)
        if item and current:
            lists[current][item["id"].lower()] = item["name"]
            continue

        count = COUNT_RE.match(stripped)
        if count:
            counts[count["key"]] = int(count["value"])

    return {
        "total": counts.get("Total apps checked"),
        "installed_count": counts.get("Installed"),
        "not_installed_count": counts.get("Not installed"),
        "stale_count": counts.get("Stale (winget only)"),
        "installed": lists["installed"],
        "stale": lists["stale"],
    }


def print_diff(title, left, right):
    only = {app_id: name for app_id, name in left.items() if app_id not in right}
    print(f"{title} ({len(only)})")
    if not only:
        print("  (none)")
    for app_id, name in only.items():
        print(f"  - {name} [{app_id}]")


first = parse_summary(FIRST_LOG)
second = parse_summary(SECOND_LOG)

print(f"First : {FIRST_LOG.name} (apps checked: {first['total']})")
print(f"Second: {SECOND_LOG.name} (apps checked: {second['total']})")

First : 2026-09-21_21-18-55.log (apps checked: 451)
Second: 2026-10-05_11-19-24.log (apps checked: 451)


## 1. Total installed

In [2]:
print(f"First : {first['installed_count']}")
print(f"Second: {second['installed_count']}")
print(f"Diff  : {second['installed_count'] - first['installed_count']:+d}")

First : 23
Second: 29
Diff  : +6


## 2. Installed in first file but not in second

In [3]:
print_diff("Installed only in first", first["installed"], second["installed"])

Installed only in first (2)
  - Windows Media Player [9wzdncrfj3pt]
  - Windows Camera [9wzdncrfjbbg]


## 3. Installed in second file but not in first

In [4]:
print_diff("Installed only in second", second["installed"], first["installed"])

Installed only in second (8)
  - Visual Studio Code [xp9khm4bk9fz7q]
  - 4K Video Downloader Plus [xp8lvtc3ltsfn3]
  - Media Player [9p5mvrnlgg2q]
  - Grammarly: AI Writing and Grammar App [xpddxx9qw8n9d7]
  - Dell Optimizer for Dell PCs [xp9b49cj91xf01]
  - GOG GALAXY [xpffxw40w60kcf]
  - Beautiful Themes Pack — Cursor & Widget Customization [9nvhplglp9cx]
  - Sucrose Wallpaper Engine [xp8jgpbhtjglcq]


## 4. Total not installed

In [5]:
print(f"First : {first['not_installed_count']}")
print(f"Second: {second['not_installed_count']}")
print(f"Diff  : {second['not_installed_count'] - first['not_installed_count']:+d}")

First : 424
Second: 415
Diff  : -9


## 5. Total stale

In [6]:
print(f"First : {first['stale_count']}")
print(f"Second: {second['stale_count']}")
print(f"Diff  : {second['stale_count'] - first['stale_count']:+d}")

First : 4
Second: 7
Diff  : +3


## 6. Stale in first file but not in second

In [7]:
print_diff("Stale only in first", first["stale"], second["stale"])

Stale only in first (0)
  (none)


## 7. Stale in second file but not in first

In [8]:
print_diff("Stale only in second", second["stale"], first["stale"])

Stale only in second (3)
  - Blue Mail - Email & Calendar [9pmhzvm588p4]
  - Visual Studio Community [xpdcfjdklzjlp8]
  - Camera· [9nkrks354g33]
